In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
while not (repo_root / "src").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
for candidate in (repo_root, repo_root / "src"):
    candidate_str = str(candidate)
    if candidate.exists() and candidate_str not in sys.path:
        sys.path.insert(0, candidate_str)


In [ ]:
from pathlib import Path
import csv
import json

from src.drive_service.logging_utils import setup_logging
from src.pipeline_paths import build_pipelines_paths
from src.turni_enrichment import enrich_pairs_by_employee


In [ ]:
root = "1FUosjKncLt18JzojmX8tKQm1nbgPI133"
paths = build_pipelines_paths(root)

pairs_name = "*.pairs.csv"
pairs_files = sorted(Path(paths.shifts_output).glob(pairs_name))
if not pairs_files:
    raise FileNotFoundError(
        f"No pairs files found in {paths.shifts_output} with pattern {pairs_name}"
    )

paths.shifts_output, paths.enrichment_output, len(pairs_files), pairs_files[:5]


In [ ]:
verbose = True
min_hours = 6.0
include_holidays = True
stats_json = paths.enrichment_output / "turni_enrichment.stats.json"

setup_logging(verbose)

stats = enrich_pairs_by_employee(
    input_dir=str(paths.shifts_output),
    output_dir=str(paths.enrichment_output),
    min_hours=min_hours,
    include_holidays=include_holidays,
)

payload = {"stats": stats}
with open(stats_json, "w", encoding="utf-8") as handle:
    json.dump(payload, handle, ensure_ascii=False, indent=2)

payload["stats"]


In [ ]:
enriched_files = sorted(Path(paths.enrichment_output).glob("*.enriched.csv"))
len(enriched_files), enriched_files[:5]


In [ ]:
if enriched_files:
    sample_enriched = enriched_files[0]
    with open(sample_enriched, "r", encoding="utf-8", newline="") as handle:
        reader = csv.reader(handle)
        sample_rows = []
        for i, row in enumerate(reader):
            sample_rows.append(row)
            if i >= 10:
                break
    sample_enriched, sample_rows
else:
    "No enriched CSV files generated"


In [ ]:
{
    "stats_json": str(stats_json),
    "stats": stats,
    "enriched_files_count": len(enriched_files),
    "enriched_files_preview": [str(path) for path in enriched_files[:3]],
}
